In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, collections
import numpy as np

PROJECT_DIR  = '/content/drive/MyDrive/PACL_Project'
PHASE3_DIR   = f'{PROJECT_DIR}/phase3_outputs'
PHASE4_DIR   = f'{PROJECT_DIR}/phase4_outputs'
os.makedirs(PHASE4_DIR, exist_ok=True)

TRAIN_JSONL      = f'{PHASE3_DIR}/train.jsonl'
TEST_OPEN_JSONL  = f'{PHASE3_DIR}/test_open.jsonl'
EMB_TRAIN        = f'{PHASE3_DIR}/train_embeddings_byt5base.npy'
EMB_OPEN         = f'{PHASE3_DIR}/test_open_embeddings_byt5base.npy'

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

print('بارگذاریِ فایل‌هایِ داده...')
train_data     = load_jsonl(TRAIN_JSONL)
test_open_data = load_jsonl(TEST_OPEN_JSONL)
print(f'train: {len(train_data):,} نمونه')
print(f'test_open: {len(test_open_data):,} نمونه')

In [ ]:
print("=== فیلدهایِ train.jsonl ===")
print(list(train_data[0].keys()))
print("نمونه:", json.dumps({k: str(v)[:60] for k, v in train_data[0].items()}, ensure_ascii=False))

print()
print("=== فیلدهایِ test_open.jsonl ===")
print(list(test_open_data[0].keys()))
print("نمونه:", json.dumps({k: str(v)[:60] for k, v in test_open_data[0].items()}, ensure_ascii=False))

HAVE_SCHOOL       = "school" in test_open_data[0]
HAVE_RHYTHM       = "rhythm" in test_open_data[0]
HAVE_RADIF        = "radif"  in test_open_data[0] or "has_radif" in test_open_data[0]
TRAIN_HAVE_SCHOOL = "school" in train_data[0]
TRAIN_HAVE_RHYTHM = "rhythm" in train_data[0]

print()
print(f"test_open دارایِ فیلدِ school : {HAVE_SCHOOL}")
print(f"test_open دارایِ فیلدِ rhythm : {HAVE_RHYTHM}")
print(f"test_open دارایِ فیلدِ radif   : {HAVE_RADIF}")
print(f"train     دارایِ فیلدِ school : {TRAIN_HAVE_SCHOOL}")
print(f"train     دارایِ فیلدِ rhythm : {TRAIN_HAVE_RHYTHM}")

if not HAVE_SCHOOL or not HAVE_RHYTHM:
    print()
    print("⚠️  فیلد school یا rhythm در test_open وجود ندارد.")
    print("   بدون آن‌ها، آزمایش نمی‌تواند اجرا شود.")
    raise RuntimeError("Missing required fields in test_open.jsonl")

In [ ]:
poet_to_school  = {}
poet_to_rhythm  = {}

for rec in train_data:
    p = rec.get("poet", "")
    s = rec.get("school", "")
    r = rec.get("rhythm", "")
    if p and s and p not in poet_to_school:
        poet_to_school[p] = s
    if p and r and p not in poet_to_rhythm:
        poet_to_rhythm[p] = r

print(f"poet_to_school: {len(poet_to_school)} شاعر  ->  {len(set(poet_to_school.values()))} مکتب")
print("نمونه:", dict(list(poet_to_school.items())[:5]))

POET_SCHOOL_PATH = f"{PHASE4_DIR}/poet_to_school.json"
with open(POET_SCHOOL_PATH, "w", encoding="utf-8") as f:
    json.dump(poet_to_school, f, ensure_ascii=False, indent=2)
print(f"✅ ذخیره شد: {POET_SCHOOL_PATH}")

gallery_poets   = [rec["poet"]                   for rec in train_data]
gallery_schools = [rec.get("school", "")         for rec in train_data]
gallery_rhythms = [rec.get("rhythm", "")         for rec in train_data]
print(f"gallery ساخته شد: {len(gallery_poets):,} آیتم")

In [ ]:
import time
import numpy as np

print("بارگذاری embeddingها...")
emb_train = np.load(EMB_TRAIN)
emb_open  = np.load(EMB_OPEN)
print(f"emb_train: {emb_train.shape}  |  emb_open: {emb_open.shape}")

def l2_norm(x):
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.clip(n, 1e-12, None)

emb_open_n = l2_norm(emb_open.astype(np.float32))
print("✅ normalize شد.")

first_rec = test_open_data[0]
has_k_field = ("k" in first_rec or "num_shots" in first_rec or "shots" in first_rec)
if has_k_field:
    def get_k(rec):
        val = rec.get("k", rec.get("num_shots", rec.get("shots", -1)))
        try:
            return int(val)
        except (TypeError, ValueError):
            return -1
    flat_k1 = [(i, rec) for i, rec in enumerate(test_open_data) if get_k(rec) == 1]
    print(f"فیلد k یافت شد (نوع: {type(test_open_data[0].get('k')).__name__}). k=1: {len(flat_k1):,} نمونه")
else:
    flat_k1 = list(enumerate(test_open_data))
    print(f"فیلد k وجود ندارد -> همه {len(flat_k1):,} نمونه به‌عنوان k=1")

k1_global_indices = np.array([x[0] for x in flat_k1], dtype=np.int64)
k1_records        = [x[1] for x in flat_k1]

gallery_emb      = emb_open_n[k1_global_indices]
gallery_poets    = np.array([rec["poet"]              for rec in k1_records])
gallery_poem_ids = np.array([str(rec["poem_id"])      for rec in k1_records])
gallery_rhythms  = [rec.get("rhythm", "")   for rec in k1_records]
gallery_schools  = [rec.get("school", "")   for rec in k1_records]
N_gallery = len(gallery_poets)
print(f"gallery (test_open k=1): {N_gallery:,} شعر")
unique_poem_ids = len(set(gallery_poem_ids))
print(f"poem_id منحصربهفرد: {unique_poem_ids:,} (میانگین بیت/شعر: {N_gallery/unique_poem_ids:.1f})")

print()
print("اجرای retrieval Open-Set k=1")
print(f"(gallery=test_open k=1، exclude_same_poem=True — دقیقاً مثل Phase 4.1)")

BATCH = 500   
all_errors   = []
byt5_hits_k1 = []

t0 = time.time()
for start in range(0, N_gallery, BATCH):
    end = min(start + BATCH, N_gallery)
    q_emb     = gallery_emb[start:end]                    
    q_poets   = gallery_poets[start:end]                 
    q_pids    = gallery_poem_ids[start:end]              

    sims = q_emb @ gallery_emb.T                         

    same_poem_mask = (q_pids[:, None] == gallery_poem_ids[None, :]) 
    sims[same_poem_mask] = -np.inf

    top1_idxs = np.argmax(sims, axis=1)                  

    for b_i in range(end - start):
        rec       = k1_records[start + b_i]
        top1_idx  = int(top1_idxs[b_i])
        true_poet = q_poets[b_i]
        pred_poet = gallery_poets[top1_idx]
        is_correct = (pred_poet == true_poet)
        byt5_hits_k1.append(1 if is_correct else 0)

        if not is_correct:
            q_rhythm         = rec.get("rhythm", "")
            retrieved_rhythm = gallery_rhythms[top1_idx]
            true_school      = poet_to_school.get(str(true_poet), rec.get("school", ""))
            pred_school      = poet_to_school.get(str(pred_poet), gallery_schools[top1_idx])
            radif_val        = rec.get("radif", rec.get("has_radif", None))
            all_errors.append({
                "true_poet":        str(true_poet),
                "pred_poet":        str(pred_poet),
                "true_school":      true_school,
                "pred_school":      pred_school,
                "query_rhythm":     q_rhythm,
                "retrieved_rhythm": retrieved_rhythm,
                "radif":            radif_val,
            })

    if (start // BATCH) % 10 == 0:
        elapsed = time.time() - t0
        print(f"  [{end:,}/{N_gallery:,}]  {elapsed:.1f}s")

byt5_hits_k1 = np.array(byt5_hits_k1, dtype=np.int8)
recall1_byt5 = float(byt5_hits_k1.mean())
print()
print(f"✅ بازیابی تمام شد.")
print(f"کل query: {len(byt5_hits_k1):,}")
print(f"تعداد خطا: {len(all_errors):,}")
print(f"Recall@1 (ByT5 Open k=1): {recall1_byt5:.4f}  (مرجع: 0.3157)")

hits_path = f"{PHASE4_DIR}/step1_byt5_per_query.npz"
np.savez_compressed(hits_path, hits=byt5_hits_k1)
print(f"✅ hits ذخیره شد: {hits_path}")

In [ ]:
print("تحلیل طبقه‌بندی خطا...")
total_errors = len(all_errors)

same_school_count = 0
same_meter_count  = 0
both_count        = 0
no_radif_count    = 0
has_radif_field   = any(e["radif"] is not None for e in all_errors)

covered_indices = set()

for i, err in enumerate(all_errors):
    ts = err["true_school"]
    ps = err["pred_school"]
    qr = err["query_rhythm"]
    rr = err["retrieved_rhythm"]
    rv = err["radif"]

    is_same_school = bool(ts and ps and ts == ps)
    is_same_meter  = bool(qr and rr and qr == rr)

    if is_same_school:
        same_school_count += 1
    if is_same_meter:
        same_meter_count += 1
    if is_same_school and is_same_meter:
        both_count += 1
    if is_same_school or is_same_meter:
        covered_indices.add(i)
    if has_radif_field and rv in (False, 0, "0", "False", "false", "no"):
        no_radif_count += 1

residual_count = total_errors - len(covered_indices)

print(f"کل خطا: {total_errors:,}")
print(f"[1] Intra-School : {same_school_count:,}  ({same_school_count/total_errors*100:.1f}%)")
print(f"[2] Shared Meter : {same_meter_count:,}  ({same_meter_count/total_errors*100:.1f}%)")
print(f"[3] هر دو (School+Meter) : {both_count:,}  ({both_count/total_errors*100:.1f}%)")
if has_radif_field:
    print(f"[4] No Radif : {no_radif_count:,}  ({no_radif_count/total_errors*100:.1f}%)")
else:
    print("[4] No Radif : N/A (فیلد radif در داده وجود ندارد)")
print(f"[5] Residual : {residual_count:,}  ({residual_count/total_errors*100:.1f}%)")

In [ ]:
def pct(n, total):
    return f"{n/total*100:.1f}"

bslash = chr(92)
pct_sign = "%"

rows = [
    "Intra-School Stylistic Confounder & " + str(same_school_count) + " & " + pct(same_school_count, total_errors) + pct_sign + " & Poets of same school share vocabulary and imagery. " + bslash*2,
    "Prosodic Meter Confounder (Aruz) & "  + str(same_meter_count)  + " & " + pct(same_meter_count, total_errors)  + pct_sign + " & Query and retrieved poem share the same prosodic meter. " + bslash*2,
    "Both School and Meter & "             + str(both_count)        + " & " + pct(both_count, total_errors)        + pct_sign + " & Co-occurrence of both constraints; strongest confounding factor. " + bslash*2,
    "Residual / Outliers & "               + str(residual_count)    + " & " + pct(residual_count, total_errors)    + pct_sign + " & Errors unexplained by school or meter. " + bslash*2,
]

latex_lines = [
    bslash + "begin{table}[t]",
    bslash + "centering",
    bslash + "caption{Quantitative Error Taxonomy for ByT5 at Open $k=1$ ($N=" + str(total_errors) + "$ misclassified queries).}",
    bslash + "label{tab:error_taxonomy}",
    bslash + "begin{tabular}{l r r p{5cm}}",
    bslash + "toprule",
    bslash + "textbf{Error Category} & " + bslash + "textbf{Count ($N$)} & " + bslash + "textbf{" + pct_sign + " of Errors} & " + bslash + "textbf{Explanation} " + bslash*4,
    bslash + "midrule",
] + rows + [
    bslash + "bottomrule",
    bslash + "end{tabular}",
    bslash + "end{table}",
]
latex_str = "\n".join(latex_lines)
print("=== LaTeX Table ===")
print(latex_str)

results = {
    "experiment": "Exp3_Error_Taxonomy",
    "spec_source": "EXP.txt (professor instructions)",
    "total_queries_k1": int(len(byt5_hits_k1)),
    "total_errors": int(total_errors),
    "recall1_byt5_open_k1": recall1_byt5,
    "taxonomy": {
        "intra_school":      {"count": int(same_school_count), "pct": round(same_school_count/total_errors*100, 2)},
        "shared_meter":      {"count": int(same_meter_count),  "pct": round(same_meter_count/total_errors*100, 2)},
        "both_school_meter": {"count": int(both_count),        "pct": round(both_count/total_errors*100, 2)},
        "residual":          {"count": int(residual_count),    "pct": round(residual_count/total_errors*100, 2)},
    },
    "has_radif_field": has_radif_field,
    "latex_table": latex_str,
}
if has_radif_field:
    results["taxonomy"]["no_radif"] = {"count": int(no_radif_count), "pct": round(no_radif_count/total_errors*100, 2)}

OUT_PATH = f"{PHASE4_DIR}/exp3_error_taxonomy_results.json"
with open(OUT_PATH, "w", encoding="utf-8") as fout:
    json.dump(results, fout, ensure_ascii=False, indent=2)
print("\n\u2705 \u0630\u062e\u06cc\u0631\u0647 \u0634\u062f: " + OUT_PATH)
print(json.dumps({k: v for k, v in results.items() if k != "latex_table"}, ensure_ascii=False, indent=2))
